In [1]:
import json
import csv
import math
import time
from pathlib import Path
from langchain_chroma import Chroma

DOCS_DIR        = "../../data/parsed_json/"
CHROMA_BASE     = "../../data/vector_db_strategy"
GOLDEN_SET_PATH = "../../eval/golden_dataset/golden_dataset_v2.csv"
COLLECTION_NAME = "rfp_docs"
EMBEDDING_MODEL = "bge-m3"
BATCH_SIZE      = 500
TOP_K           = 10

In [2]:
STRATEGIES = [
    {"name": "A_baseline",        "chunk_size": 1000, "overlap": 0,   "merge_table": False},
    {"name": "B1_overlap100",     "chunk_size": 1000, "overlap": 100, "merge_table": False},
    {"name": "B2_overlap200",     "chunk_size": 1000, "overlap": 200, "merge_table": False},
    {"name": "C_table_merge",     "chunk_size": 1000, "overlap": 0,   "merge_table": True},
    {"name": "D1_overlap100_merge","chunk_size": 1000, "overlap": 100, "merge_table": True},
    {"name": "D2_overlap200_merge","chunk_size": 1000, "overlap": 200, "merge_table": True},
]

GS_TO_DOCID = {
    "GKL_그룹웨어":            "D093",
    "KUSF_체육":               "D011",
    "강릉어선안전":             "D024",
    "경기_사회서비스":          "D087",
    "고려대학교_차세대포털":    "D008",
    "광주과기원_RCMS":          "D073",
    "광주과학기술원_학사시스템": "D039",
    "구미_육상":               "D018",
    "국립중앙의료원_응급":      "D069",
    "국민연금공단_이러닝":      "D049",
    "국민연금_멀티턴1":         "D050",
    "국민연금_멀티턴2":         "D050",
    "국민연금_멀티턴3":         "D050",
    "국방_대용량":             "D010",
    "기초과학연구원_극저온":    "D051",
    "나노종합_팹":             "D099",
    "대검찰청_홈페이지":        "D053",
    "민속박물관_아카이브":      "D090",
    "벤처협회_시스템":          "D086",
    "보험개발원_실손":          "D083",
    "봉화군_재난":             "D005",
    "부산관광_ERP":            "D037",
    "서민금융_채팅":            "D056",
    "서영대_교육":             "D045",
    "서울_디지털성범죄":        "D068",
    "서울_지도플랫폼":          "D040",
    "서울교육청_ISP":           "D084",
    "세종_인사":               "D088",
    "우즈벡_관개":             "D072",
    "울산_버스":               "D034",
    "인천_도시계획":            "D004",
    "인천_일자리":             "D030",
    "인천공항_ERP":            "D079",
    "적십자_재해복구":          "D095",
    "철도_ISP":               "D070",
    "통합정보시스템_충돌":      "D016",
    "평택_버스":               "D060",
    "해양박물관_자료":          "D066",
    "고려대_vs_광주과기원":     ["D008", "D039"],
    "버스_다중비교":            ["D034", "D060"],
    "재난안전_종합":            ["D005", "D007"],
    "철도_vs_인천_ISP":        ["D070", "D030"],
    "TEST": None, "unknown": None, "ISP_다중비교": None,
    "교육관련_다중문서": None, "문화_다중비교": None,
    "의료_다중문서": None, "재공고_종합": None,
    "신규_vs_고도화": None, "예산_최소_최대": None,
    "멀티턴_심화1": None, "멀티턴_심화2": None,
    "모른다_테스트1": None, "모른다_테스트2": None,
    "모른다_테스트3": None, "모른다_테스트4": None,
    "모른다_테스트5": None, "모른다_테스트6": None,
    "존재하지않는사업": None, "입찰마감_확인": None,
}

In [3]:
def clean(val):
    if val is None:
        return ""
    if isinstance(val, float) and math.isnan(val):
        return ""
    return val

In [4]:
def build_payload(doc: dict, section: dict, block: dict) -> dict:
    meta = doc.get("metadata", {})
    return {
        "doc_id":        str(clean(doc.get("doc_id"))),
        "file_name":     str(clean(doc.get("file_name"))),
        "source_format": str(clean(doc.get("source_format"))),
        "사업명":         str(clean(meta.get("사업명"))),
        "발주기관":       str(clean(meta.get("발주기관"))),
        "사업유형":       str(clean(meta.get("사업유형"))),
        "사업금액":       float(clean(meta.get("사업금액")) or 0.0),
        "공고번호":       str(clean(meta.get("공고번호"))),
        "공고차수":       float(clean(meta.get("공고차수")) or 0.0),
        "공개일자":       str(clean(meta.get("공개일자"))),
        "입찰참여시작일":  str(clean(meta.get("입찰참여시작일"))),
        "입찰참여마감일":  str(clean(meta.get("입찰참여마감일"))),
        "재공고여부":     bool(meta.get("재공고여부", False)),
        "linked_doc_id": str(clean(meta.get("linked_doc_id"))),
        "사업요약":       str(clean(meta.get("사업요약"))),
        "header_path":   " > ".join(section.get("header_path", [])),
        "section_id":    str(clean(section.get("section_id"))),
        "block_id":      str(clean(block.get("block_id"))),
        "block_type":    str(clean(block.get("type"))),
        "table_type":    str(clean(block.get("table_type"))),
    }

In [5]:
def chunk_section(section: dict, chunk_size: int, overlap: int, merge_table: bool) -> list[dict]:
    header_prefix = " > ".join(section.get("header_path", []))
    results = []
    current_text = ""
    last_text_block = None
    prev_text = ""

    for block in section.get("blocks", []):
        if block.get("is_decorative"):
            continue

        if block["type"] == "table":
            if merge_table:
                combined = (current_text + "\n\n" + block["content"]).strip()
                if len(combined) > chunk_size and current_text.strip():
                    results.append({
                        "content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}",
                        "block":   last_text_block,
                    })
                    results.append({
                        "content": f"[섹션: {header_prefix}]\n\n{block['content']}",
                        "block":   block,
                    })
                    current_text = ""
                    last_text_block = None
                else:
                    current_text = combined + "\n\n"
                    last_text_block = block
            else:
                if current_text.strip():
                    results.append({
                        "content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}",
                        "block":   last_text_block,
                    })
                    prev_text = current_text[-overlap:] if overlap else ""
                    current_text = ""
                    last_text_block = None
                results.append({
                    "content": f"[섹션: {header_prefix}]\n\n{block['content']}",
                    "block":   block,
                })
        else:
            if len(current_text) + len(block["content"]) > chunk_size and current_text.strip():
                results.append({
                    "content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}",
                    "block":   last_text_block,
                })
                prev_text = current_text[-overlap:] if overlap else ""
                current_text = prev_text + block["content"] + "\n\n"
            else:
                current_text += block["content"] + "\n\n"
            last_text_block = block

    if current_text.strip():
        results.append({
            "content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}",
            "block":   last_text_block,
        })

    return results

In [6]:
def process_doc(doc: dict, chunk_size: int, overlap: int, merge_table: bool) -> tuple[list[str], list[dict]]:
    texts, metas = [], []
    meta = doc.get("metadata", {})
    name = str(clean(meta.get("사업명")))
    org  = str(clean(meta.get("발주기관")))
    prefix = f"[사업명] {name}\n[발주기관] {org}\n\n"  # A_short 방식

    for section in doc.get("sections", []):
        chunks = chunk_section(section, chunk_size, overlap, merge_table)
        for item in chunks:
            texts.append(prefix + item["content"])
            metas.append(build_payload(doc, section, item["block"] or {}))
    return texts, metas

In [7]:
def load_embedding_model(name: str):
    if name == "bge-m3":
        from langchain_huggingface import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(
            model_name="BAAI/bge-m3",
            model_kwargs={"device": "cuda"},
            encode_kwargs={"normalize_embeddings": True},
        )
    elif name == "text-embedding-3-small":
        from langchain_openai import OpenAIEmbeddings
        return OpenAIEmbeddings(model="text-embedding-3-small")
    else:
        raise ValueError(f"알 수 없는 임베딩 모델: {name}")

In [8]:
def save_vectorstore_for_strategy(strategy: dict, all_texts: list, all_metas: list, embedding_model):
    chroma_dir = f"{CHROMA_BASE}/{strategy['name']}"

    if Path(chroma_dir).exists():
        vs = Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embedding_model,
            persist_directory=chroma_dir,
        )
        existing_count = vs._collection.count()
        if existing_count >= len(all_texts):
            print(f"[SKIP] {strategy['name']} — 완료된 DB 존재 ({existing_count}개 청크)")
            return vs
        print(f"[RESUME] {strategy['name']} — {existing_count}/{len(all_texts)}개, 이어서 진행")
        start_from = existing_count
    else:
        vs = Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embedding_model,
            persist_directory=chroma_dir,
        )
        start_from = 0

    for start in range(start_from, len(all_texts), BATCH_SIZE):
        end = start + BATCH_SIZE
        vs.add_texts(
            texts=all_texts[start:end],
            metadatas=all_metas[start:end],
        )
        print(f"  저장 완료: {min(end, len(all_texts))}/{len(all_texts)}")

    print(f"  완료 ({vs._collection.count()}개 청크) → {chroma_dir}")
    return vs

In [9]:
def hit(retrieved_ids, target_ids, k):
    return any(tid in retrieved_ids[:k] for tid in target_ids)

In [10]:
# 골든셋 로드
golden_set = []
with open(GOLDEN_SET_PATH, encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        golden_set.append(row)

golden_set = golden_set[:101]
print(f"골든셋 문항 수: {len(golden_set)}")

골든셋 문항 수: 101


In [11]:
# 문서 로드
json_files = sorted(Path(DOCS_DIR).glob("*.json"))
print(f"JSON 파일 수: {len(json_files)}")

embedding_model = load_embedding_model(EMBEDDING_MODEL)


JSON 파일 수: 97


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [12]:
# 전략별 실험
import warnings
warnings.filterwarnings("ignore", message="Relevance scores must be between")

eval_results = []

for strategy in STRATEGIES:
    print(f"\n{'='*50}")
    print(f"[전략] {strategy['name']}")

    all_texts, all_metas = [], []
    for json_file in json_files:
        with open(json_file, encoding="utf-8") as f:
            doc = json.load(f)
        texts, metas = process_doc(
            doc,
            chunk_size=strategy["chunk_size"],
            overlap=strategy["overlap"],
            merge_table=strategy["merge_table"],
        )
        all_texts.extend(texts)
        all_metas.extend(metas)

    print(f"  총 청크: {len(all_texts)}개")

    vs = save_vectorstore_for_strategy(strategy, all_texts, all_metas, embedding_model)

    recall3, recall5, recall10, mrr_scores, query_times = [], [], [], [], []
    skipped = 0

    for row in golden_set:
        gs_key = str(row["doc_id"])
        target = GS_TO_DOCID.get(gs_key)

        if target is None:
            skipped += 1
            continue

        target_ids = target if isinstance(target, list) else [target]

        t0 = time.time()
        hits = vs.similarity_search_with_relevance_scores(row["question"], k=TOP_K)
        query_times.append(time.time() - t0)

        retrieved_doc_ids = [doc.metadata.get("doc_id", "") for doc, _ in hits]

        recall3.append(1.0 if hit(retrieved_doc_ids, target_ids, 3) else 0.0)
        recall5.append(1.0 if hit(retrieved_doc_ids, target_ids, 5) else 0.0)
        recall10.append(1.0 if hit(retrieved_doc_ids, target_ids, 10) else 0.0)

        rank = next(
            (i + 1 for i, d in enumerate(retrieved_doc_ids) if d in target_ids),
            None,
        )
        mrr_scores.append(1.0 / rank if rank else 0.0)

    n = len(recall3)
    print(f"  평가: {n}개 | 제외: {skipped}개")

    eval_results.append({
        "전략":       strategy["name"],
        "Recall@3":   round(sum(recall3)    / n, 4),
        "Recall@5":   round(sum(recall5)    / n, 4),
        "Recall@10":  round(sum(recall10)   / n, 4),
        "MRR":        round(sum(mrr_scores) / n, 4),
        "avg_ms":     round(sum(query_times) / n * 1000, 1),
        "청크수":     len(all_texts),
    })


[전략] A_baseline
  총 청크: 17382개
  저장 완료: 500/17382
  저장 완료: 1000/17382
  저장 완료: 1500/17382
  저장 완료: 2000/17382
  저장 완료: 2500/17382
  저장 완료: 3000/17382
  저장 완료: 3500/17382
  저장 완료: 4000/17382
  저장 완료: 4500/17382
  저장 완료: 5000/17382
  저장 완료: 5500/17382
  저장 완료: 6000/17382
  저장 완료: 6500/17382
  저장 완료: 7000/17382
  저장 완료: 7500/17382
  저장 완료: 8000/17382
  저장 완료: 8500/17382
  저장 완료: 9000/17382
  저장 완료: 9500/17382
  저장 완료: 10000/17382
  저장 완료: 10500/17382
  저장 완료: 11000/17382
  저장 완료: 11500/17382
  저장 완료: 12000/17382
  저장 완료: 12500/17382
  저장 완료: 13000/17382
  저장 완료: 13500/17382
  저장 완료: 14000/17382
  저장 완료: 14500/17382
  저장 완료: 15000/17382
  저장 완료: 15500/17382
  저장 완료: 16000/17382
  저장 완료: 16500/17382
  저장 완료: 17000/17382
  저장 완료: 17382/17382
  완료 (17382개 청크) → ../../data/vector_db_strategy/A_baseline
  평가: 84개 | 제외: 17개

[전략] B1_overlap100
  총 청크: 17406개
  저장 완료: 500/17406
  저장 완료: 1000/17406
  저장 완료: 1500/17406
  저장 완료: 2000/17406
  저장 완료: 2500/17406
  저장 완료: 3000/17406
  저장 완료: 3500/17406

In [13]:
from IPython.display import Markdown, display

header = "| 전략 | Recall@3 | Recall@5 | Recall@10 | MRR | avg_ms | 청크수 |\n|------|----------|----------|-----------|-----|--------|--------|"
rows = "\n".join(
    f"| {r['전략']} | {r['Recall@3']} | {r['Recall@5']} | {r['Recall@10']} | {r['MRR']} | {r['avg_ms']} | {r['청크수']} |"
    for r in eval_results
)
display(Markdown(f"## 청킹 전략 실험 결과\n\n{header}\n{rows}"))

## 청킹 전략 실험 결과

| 전략 | Recall@3 | Recall@5 | Recall@10 | MRR | avg_ms | 청크수 |
|------|----------|----------|-----------|-----|--------|--------|
| A_baseline | 0.2381 | 0.2738 | 0.2976 | 0.2159 | 25.3 | 17382 |
| B1_overlap100 | 0.2262 | 0.2738 | 0.2976 | 0.2151 | 24.8 | 17406 |
| B2_overlap200 | 0.25 | 0.2738 | 0.2976 | 0.223 | 25.2 | 17436 |
| C_table_merge | 0.2143 | 0.2619 | 0.3452 | 0.1853 | 24.8 | 11375 |
| D1_overlap100_merge | 0.2262 | 0.2857 | 0.3571 | 0.193 | 24.6 | 11435 |
| D2_overlap200_merge | 0.2381 | 0.2857 | 0.3571 | 0.1927 | 25.6 | 11501 |

D1_overlap100_merge = D2_overlap200_merge > C_table_merge > A_baseline = B1 = B2

**1. 표+텍스트 병합이 Recall@10에서 효과적**
- 표 단독 청크 방식(A, B계열)은 Recall@10 0.2976으로 모두 동일
- 표+텍스트 병합(C, D계열)은 Recall@10 0.3452~0.3571로 16~20% 향상
- RFP 문서 특성상 핵심 요구사항이 표 형태로 많이 담겨 있어 앞 텍스트와 병합하면 문맥 보존 효과

**2. Overlap은 Recall@10에 영향 없음**
- A(0.2976) vs B1(0.2976) vs B2(0.2976) 모두 동일
- 청크 경계 문제보다 표 분리 문제가 더 큰 원인이었음
- 단, Recall@3 기준으로는 B2(0.2500)가 소폭 우세

**3. MRR은 baseline이 가장 높음**
- A_baseline MRR(0.2159)이 표 병합 전략보다 높음
- 표 병합 시 청크가 길어져 정답이 상위 순위에 배치되기 어려운 경향

**4. 표 병합 시 청크 수 감소**
- A계열: ~17,400개 → C/D계열: ~11,400개 (약 35% 감소)
- 청크 수 감소로 검색 속도 향상 및 저장 비용 절감 효과


### 지표 비교

| 전략 | Recall@3 | Recall@5 | Recall@10 | MRR | 청크수 |
|------|----------|----------|-----------|-----|--------|
| A_baseline | 0.2381 | 0.2738 | 0.2976 | 0.2159 | 17382 |
| B1_overlap100 | 0.2262 | 0.2738 | 0.2976 | 0.2151 | 17406 |
| B2_overlap200 | 0.2500 | 0.2738 | 0.2976 | 0.2230 | 17436 |
| C_table_merge | 0.2143 | 0.2619 | 0.3452 | 0.1853 | 11375 |
| D1_overlap100_merge | 0.2262 | 0.2857 | 0.3571 | 0.1930 | 11435 |
| D2_overlap200_merge | 0.2381 | 0.2857 | 0.3571 | 0.1927 | 11501 |

### 결론
D1(overlap=100, merge_table=True) 채택 — Recall@10 최고(0.3571)이며 overlap 100과 200의 차이가 미미하므로 청크 수가 더 적은 D1이 효율적. 단, MRR이 낮아 정답이 상위 순위에 배치되기 어려운 경향이 있으므로 실서비스에서 Top-K를 충분히 크게 설정하거나 재순위화(reranker) 도입을 고려할 것.